# 01 — Generate monthly geomedian and rolling NDVI

Build a cloud-masked Landsat NDVI time series for a polygon study area. The filename is retained for continuity with the supplied research workflow.


## Inputs, outputs, and execution order

**Inputs:** A polygon boundary; access to `ls5_sr`, `ls7_sr`, `ls8_sr`, and `ls9_sr` in a configured Open Data Cube.

**Outputs:** Monthly `data/ndvi_roll3_YYYYMM_v01.tif` rasters, used by notebooks 02 and 04.

Run cells from top to bottom in a fresh kernel. Open this notebook beside `workflow_config.json` and `workflow_support.py`. The configuration uses Mion study settings; the original boundary and final criteria table were not supplied. See the README for the small example and for adapting the workflow.

### How to interpret this step

The code computes a geomedian of red and near-infrared reflectance **within each month**, then a centered three-month median of each reflectance band, and finally NDVI. It does not compute the full GeoMAD deviation layers or a single geomedian across every observation in a three-month window. Monthly timestamps represent month starts. Missing observations can remain missing; `min_periods=1` permits partial support.


## 1. Import packages

Use the Digital Earth Africa Sandbox for steps that access the data cube. Local raster processing uses the packages listed in `environment.yml`.


In [ ]:
%matplotlib inline
import os
import gc
import psutil
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray
import datacube
from odc.algo import xr_geomedian
from deafrica_tools.datahandling import load_ard
from deafrica_tools.bandindices import calculate_indices
from deafrica_tools.spatial import xr_rasterize
from dask.diagnostics import ProgressBar
ProgressBar().register()


## 2. Load shared settings

The settings below describe the run. Edit the JSON configuration before continuing; later parameter cells explain the settings used by this notebook.


In [ ]:
# Edit workflow_config.json once, then restart and rerun each notebook.
from workflow_support import load_config, require_files, load_aoi, assert_same_grid
cfg = load_config()
print(f"Study: {cfg['study_id']} | Years: {cfg['start_year']}-{cfg['end_year']}")
print(f"Output directory: {cfg['output_dir']}")


## 3. Connect to the data cube

A Python installation alone does not provide the DE Africa index. Confirm that your environment has the required products and overlapping datasets.


In [ ]:
dc = datacube.Datacube(app='Generating_geomedian_composites')


## 4. Read the study boundary

The polygon masks pixels; its bounding box defines the data query. All features in the supplied boundary are included.


In [ ]:
# The helper merges all polygon features and converts their CRS to EPSG:4326.
# Supply your own GeoJSON, GeoPackage, or complete shapefile in the configuration.
geopolygon_gdf = load_aoi(cfg["aoi_vector"])
minx, miny, maxx, maxy = geopolygon_gdf.total_bounds
lat_range = (miny, maxy)
lon_range = (minx, maxx)
print("Query bounds (longitude, latitude):", lon_range, lat_range)


## 5. Set processing parameters

`EPSG:6933` supplies an equal-area grid in metres. Resolution is `(y, x)`; `(-30, 30)` means 30 m north-up pixels. Start with a small polygon and one year, then restore the complete time range required by the indicators. Dask chunks control spatial blocks, not a guaranteed RAM limit.


In [ ]:
folder_storage = cfg["output_dir"]
output_folder = f"{folder_storage}/data"
os.makedirs(output_folder, exist_ok=True)
output_crs = cfg["output_crs"]
resolution = tuple(cfg["resolution"])
start_year, end_year = cfg["start_year"], cfg["end_year"]
product, stat, version = "ndvi", "roll3", cfg["file_version"]
dask_chunks = cfg["dask_chunks"]


## 6. Monitor memory

Resident memory reports help diagnose large loads; garbage collection does not impose a memory ceiling.


In [ ]:
# =========================================================
# FUNCTION: PRINT RAM USAGE
# =========================================================
def print_ram(stage=""):
    process = psutil.Process(os.getpid())
    mem_gb = process.memory_info().rss / (1024 ** 3)
    print(f"[RAM] {stage}: {mem_gb:.2f} GB")


## 7. Generate and save monthly rasters

The December/January buffer supports centered smoothing at year boundaries. Existing files are skipped only when configured. Confirm 12 monthly files per requested year before continuing; file existence does not guarantee valid pixel coverage.


In [ ]:
# =========================================================
# YEARLY PROCESSING
# =========================================================
failed_years = []
for yr in range(start_year, end_year + 1):
    print("\n" + "=" * 70)
    print(f"Processing year: {yr}")
    print("=" * 70)
    print_ram("Start of year")

    try:
        # -------------------------------------------------
        # Load only the buffered period needed for roll-3
        # -------------------------------------------------
        time_period = (f"{yr-1}-12-01", f"{yr+1}-01-31")

        ds = load_ard(
            dc=dc,
            products=cfg["landsat_products"],
            measurements=["red", "nir"],   # only what NDVI needs
            mask_filters=[("opening", 3), ("dilation", 3)],
            time=time_period,
            x=lon_range,
            y=lat_range,
            output_crs=output_crs,
            resolution=resolution,
            dask_chunks=dask_chunks,
        )
        print_ram("After load_ard")

        if ds is None or "time" not in ds.dims or ds.time.size == 0:
            failed_years.append(yr)
            print(f"No data found for year {yr}")
            del ds
            gc.collect()
            print_ram("After cleanup - no data")
            continue

        # -------------------------------------------------
        # Apply AOI mask early
        # -------------------------------------------------
        mask = xr_rasterize(geopolygon_gdf, ds)
        ds = ds.where(mask)
        print_ram("After AOI masking")

        # -------------------------------------------------
        # Monthly geomedian
        # -------------------------------------------------
        # One red/NIR geomedian per calendar month; this is not a MAD-layer calculation.
        monthly_geomad = ds.resample(time="1MS").map(xr_geomedian)
        print_ram("After monthly geomedian")

        # -------------------------------------------------
        # 3-month rolling median
        # -------------------------------------------------
        # Centered window includes the preceding and following months.
        # Band-wise rolling medians are computed before NDVI; this changes temporal support.
        roll3 = monthly_geomad.rolling(time=3, min_periods=1, center=True).median()
        print_ram("After rolling median")

        # -------------------------------------------------
        # NDVI
        # -------------------------------------------------
        ndvi = calculate_indices(
            roll3,
            index="NDVI",
            satellite_mission="ls"
        ).NDVI
        print_ram("After NDVI calculation")

        # -------------------------------------------------
        # Keep only the target year
        # -------------------------------------------------
        ndvi_year = ndvi.sel(time=slice(f"{yr}-01-01", f"{yr}-12-31"))
        print_ram("After selecting target year")

        if "time" not in ndvi_year.dims or ndvi_year.time.size == 0:
            failed_years.append(yr)
            print(f"No outputs to save for year {yr}")
            del ds, mask, monthly_geomad, roll3, ndvi, ndvi_year
            gc.collect()
            print_ram("After cleanup - no outputs")
            continue

        # -------------------------------------------------
        # CRS
        # -------------------------------------------------
        crs = ds.rio.crs
        if crs is None:
            raise ValueError(f"No CRS found for year {yr}")

        # -------------------------------------------------
        # Save one month at a time
        # -------------------------------------------------
        saved_count = 0

        for i, t in enumerate(ndvi_year.time.values):
            timestamp = pd.to_datetime(str(t))
            yyyymm = f"{timestamp.year}{timestamp.month:02d}"

            filename = f"{product}_{stat}_{yyyymm}_{version}.tif"
            out_path = os.path.join(output_folder, filename)

            if cfg["skip_existing"] and os.path.exists(out_path):
                print(f"Skipping existing file: {out_path}")
                continue

            print(f"Saving {filename} ...")
            da = ndvi_year.isel(time=i).rio.write_crs(crs)

            # Preserve valid zero NDVI; use NaN explicitly for missing pixels.
            da = da.rio.write_nodata(np.nan)
            da.rio.to_raster(out_path)
            print(f"Saved: {out_path}")
            print_ram(f"After saving {filename}")

            del da
            gc.collect()

            saved_count += 1

        print(f"✓ Finished year {yr}. Files saved: {saved_count}")

        # -------------------------------------------------
        # Clean up memory before next year
        # -------------------------------------------------
        del ds, mask, monthly_geomad, roll3, ndvi, ndvi_year
        gc.collect()
        print_ram("End of year cleanup")

    except Exception as e:
        failed_years.append(yr)
        print(f"Error processing year {yr}: {e}")
        gc.collect()
        print_ram("After exception cleanup")

if failed_years:
    raise RuntimeError(f"Incomplete monthly outputs for years: {failed_years}. Review the messages above.")
print("\n✓ All requested years processed; check monthly output counts.")


## Check the outputs before continuing

- Review warnings and errors, and confirm that every requested year or reporting period was written.
- Open an output raster and inspect its CRS, extent, nodata mask, and value range.
- Keep the configuration and software versions with your results. Run `python scripts/capture_environment.py` from the repository root.
- If you change the AOI, dates, metric, or thresholds, use a fresh output directory to avoid reusing older results.

See [README.md](README.md), [data requirements](docs/DATA_REQUIREMENTS.md), and [method details](docs/METHODS.md).

The supplied notebooks identify their code as Apache License 2.0; retain the original Digital Earth Africa attribution when reusing it. Dataset licences and acknowledgements apply separately. Use the Issues tab of the repository hosting this workflow for workflow-specific questions.

**Validation status:** documentation, syntax, and synthetic-data checks are described in `docs/VALIDATION.md`. This revised workflow has not been executed against the live Digital Earth Africa data cube. A displayed current date or package version is not evidence of a successful full run.
